In [ ]:
%load_ext autoreload
%autoreload 2

from highres_ta.preprocessing import load_data, preprocess_data, load_config, train_test_split, ALL_FEATURES, load_preprocessed_df
from highres_ta.utils import ROOT

import pandas as pd

config = load_config(ROOT / "scripts/example_config.yaml")
config.xname_features = ALL_FEATURES

# Preprocessed train-test data with all features

In [ ]:
raw_df = load_data()
raw_df

In [ ]:
df_na = preprocess_data(raw_df.copy(), config, dropna=False)
df_na

In [ ]:
# export to parquet
df_na.to_parquet(ROOT / "data/preprocessed_train_datasets/df_all_features.pq", index=True)


In [ ]:
import pandas as pd
df_loaded = pd.read_parquet(ROOT / "data/preprocessed_train_datasets/df_all_features.pq")
df_loaded

In [ ]:
train_x, train_y, test_x, test_y = train_test_split(
    df = df_na, 
    config = config
)

In [ ]:
train_df = train_x.copy()
train_df['talk'] = train_y


test_df = test_x.copy()
test_df['talk'] = test_y

In [ ]:
train_df

In [ ]:
train_df.to_parquet(ROOT / "data/preprocessed_train_datasets/train_df_all_features.pq", index=True)
test_df.to_parquet(ROOT / "data/preprocessed_train_datasets/test_df_all_features.pq", index=True)

In [ ]:
train_df_loaded = load_preprocessed_df(ROOT / "data/preprocessed_train_datasets/train_df_all_features.pq", ALL_FEATURES, dropna=False)
train_df_loaded

# Catboost structural run 4 datasets



In [ ]:
from highres_ta import estimators as models


MODEL_NAME = "bagged_catboost_residual_modelstructural_optuna_4"
trained_model = (
    models.BaggingCatBoostResidualRegressor.load(
        ROOT
        / f"models/{MODEL_NAME}.pkl"
    )
)


data_save_path = ROOT/f"data/{MODEL_NAME}"



model_config = load_config(
        ROOT / "scripts/example_config.yaml"
    )

model_config.xname_features = (
        trained_model.feature_names_in_.tolist()
    )

In [ ]:
df = load_preprocessed_df(
    ROOT / "data/preprocessed_train_datasets_all_features/df_all_features.pq", 
    model_config.xname_features, 
    dropna=True
)

train_df = load_preprocessed_df(
    ROOT / "data/preprocessed_train_datasets_all_features/train_df_all_features.pq",
    model_config.xname_features,
    dropna=True
)

test_df = load_preprocessed_df(
    ROOT / "data/preprocessed_train_datasets_all_features/test_df_all_features.pq",
    model_config.xname_features,            
    dropna=True
)


df.to_parquet(data_save_path/"catboost_optuna_4_df.pq", index=True)
train_df.to_parquet(data_save_path/"catboost_optuna_4_train_df.pq", index=True)
test_df.to_parquet(data_save_path/"catboost_optuna_4_test_df.pq", index=True)


In [ ]:
def make_swapped_df(raw_df, swapped_salinity_name:str, config):

    from copy import deepcopy
    
    raw_df = load_data()

    keys = ["expocode", "time", "lat", "lon", "depth"]

    valid_original = raw_df[
    raw_df["salinity"].notna()
    ]

    valid_swapped = raw_df[
    raw_df[swapped_salinity_name].notna()
    ]

    common = (
    valid_original[keys]
    .merge(valid_swapped[keys], on=keys)
    .drop_duplicates()
    )

    raw_common = raw_df.merge(common, on=keys)

    keys = ["expocode", "time", "lat", "lon", "depth"]

    raw_common= (
    raw_common
    .groupby(keys, as_index=False)
    .mean(numeric_only=True)
    )

    # 1. Run preprocessing on both (which assigns the multi-index)
    original_df = preprocess_data(raw_common.copy(), config, dropna = True)

    swapped_config = deepcopy(config)
    swapped_config.salinity_name = swapped_salinity_name
    swapped_df = preprocess_data(raw_common.copy(), swapped_config, dropna = True)
    
        # 2. Drop the volatile index levels so both DFs share the exact same 4-level index 
    levels_to_drop = ["salinity_bin", "is_coastal"] 
    original_core = original_df.reset_index(level=levels_to_drop) 
    swapped_core = swapped_df.reset_index(level=levels_to_drop) 

    # 3. Find the exact matching rows using a basic index intersection 
    common_idx = original_core.index.intersection(swapped_core.index) 

    # 4. Use .loc to slice both dataframes to the exact same rows in the exact same order
    original_df = original_core.loc[common_idx]
    swapped_df = swapped_core.loc[common_idx]
    
    
    return original_df, swapped_df




    
    
    
    

In [ ]:
import os

for salinity_name in ["sss_cci", "salt_soda", "sss_glorys", "sss_multiobs"]:
    
    original_df, swapped_df = make_swapped_df(salinity_name, model_config)
    
    save_dir = data_save_path / "salinity_swap" / salinity_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    original_df.to_parquet(
        data_save_path/f"salinity_swap/{salinity_name}/original_df.pq"
    )
    swapped_df.to_parquet(
        data_save_path/f"salinity_swap/{salinity_name}/swapped_df.pq"
    )

In [ ]:
def make_swapped_test(swapped_salinity_name: str, MODEL_NAME=MODEL_NAME):

    original_test_df = pd.read_parquet(
        ROOT / f"data/{MODEL_NAME}/catboost_optuna_4_test_df.pq"
    )

    original_df = pd.read_parquet(
        ROOT / f"data/{MODEL_NAME}/salinity_swap/{swapped_salinity_name}/original_df.pq"
    )

    swapped_df = pd.read_parquet(
        ROOT / f"data/{MODEL_NAME}/salinity_swap/{swapped_salinity_name}/swapped_df.pq"
    )

    # --------------------------------------------------------
    # align only on stable index levels
    # --------------------------------------------------------

    join_levels = ["expocode", "time", "lat", "lon", "depth"]

    original_test_core = (
        original_test_df
        .reset_index()
        .set_index(join_levels)
    )

    original_core = (
        original_df
        .reset_index()
        .set_index(join_levels)
    )

    swapped_core = (
        swapped_df
        .reset_index()
        .set_index(join_levels)
    )

    common_idx = original_core.index.intersection(original_test_core.index)

    original_test_df = original_core.loc[common_idx].copy()
    swapped_test_df = swapped_core.loc[common_idx].copy()

    return original_test_df, swapped_test_df

In [ ]:
for salinity_name in ["sss_cci", "salt_soda", "sss_glorys", "sss_multiobs"]:
    
    original_test_df, swapped_test_df = make_swapped_test(salinity_name)
    
    save_dir = data_save_path / "salinity_swap" / salinity_name
    
    original_test_df.to_parquet(
        data_save_path/f"salinity_swap/{salinity_name}/original_test_df.pq"
    )
    swapped_test_df.to_parquet(
        data_save_path/f"salinity_swap/{salinity_name}/swapped_test_df.pq"
    )

In [ ]:
from functools import reduce
from os import mkdir
import pandas as pd

base_dir = ROOT/"data/bagged_catboost_residual_modelstructural_optuna_4/salinity_swap"

salinity_products = [
    "sss_cci",
    "salt_soda",
    "sss_glorys",
    "sss_multiobs",
]


swapped_dfs = {}

for product in salinity_products:

    swapped_df = pd.read_parquet(
        base_dir / product / "swapped_df.pq"
    )
    
    print(f"{product:<15} -> {len(swapped_df):,} samples")

    swapped_dfs[product] = swapped_df

common_index = reduce(
    pd.Index.intersection,
    [df.index for df in swapped_dfs.values()],
)

print(f"Common intersection across all products: {len(common_index):,} samples")

original_df = pd.read_parquet(
        base_dir / "salt_soda/original_df.pq"
    )

original_common = original_df.loc[common_index].sort_index()

common_dir = base_dir / "common_intersection"
common_dir.mkdir(exist_ok=True)
original_common.to_parquet(
        base_dir / "common_intersection/original_df.pq"
    )
    
for product in salinity_products:

    swapped_df = pd.read_parquet(
        base_dir / product / "swapped_df.pq"
    )

    swapped_common = swapped_df.loc[common_index].sort_index()

    # sanity checks
    assert original_common.index.equals(swapped_common.index)
    assert len(original_common) == len(common_index)

    swapped_common.to_parquet(
        base_dir / f"common_intersection/{product}_swapped_df.pq"
    )

    print(
        f"{product:<15} -> {len(common_index):,} samples saved"
    )
